# Aprendizaje por Refuerzo: Nim con Palillos

En este cuadernillo aplicamos los principios básicos del **aprendizaje por refuerzo (axr)** para entrenar un agente que aprenda a jugar al **Nim**, un juego matemático clásico de dos jugadores.

---

## El juego: Nim con palillos

Las reglas son muy simples:

1. Se empieza con **N palillos** sobre la mesa (por defecto, 15).
2. En cada turno, el jugador activo puede tomar **1, 2 o 3 palillos**.
3. **Gana quien tome el último palillo.**

A pesar de su simplicidad, el juego tiene una estrategia óptima perfectamente definida: el jugador que empieza pierde si y solo si N es múltiplo de 4. Esto nos permite verificar después que el agente realmente aprendió a jugar bien.

---

## Formulación como problema de axr

Usaremos la misma aproximación que en el ejemplo del tres en raya:

- **Estado** $S$: número de palillos restantes en la mesa.
- **Acciones**: tomar 1, 2 o 3 palillos.
- **Función de valor** $V(S)$: estimación de la probabilidad de ganar desde el estado $S$.
- **Actualización TD**: al final de cada partida, propagamos la recompensa hacia atrás:

$$V(S_t) \leftarrow V(S_t) + \alpha \left[ V(S_{t+1}) - V(S_t) \right]$$

- **Exploración vs explotación**: con probabilidad `prob_exp` el agente elige una acción aleatoria; en caso contrario, elige la que lleva al estado con mayor valor.


## 1. El tablero: clase `NimBoard`

La clase `NimBoard` representa el estado del juego: cuántos palillos quedan y qué movimientos son válidos.

In [ ]:
import numpy as np


class NimBoard():
    def __init__(self, n=15):
        self.n = n          # palillos iniciales
        self.state = n      # palillos restantes (el estado observable)

    def valid_moves(self):
        """Devuelve la lista de palillos que se pueden tomar (1, 2 o 3, sin pasarse)."""
        return list(range(1, min(4, self.state + 1)))

    def update(self, action):
        """Aplica un movimiento: resta `action` palillos del estado."""
        if action not in self.valid_moves():
            raise ValueError(f"Movimiento ilegal: {action}. Válidos: {self.valid_moves()}")
        self.state -= action

    def is_game_over(self):
        """Devuelve True si ya no quedan palillos (el último en mover ganó)."""
        return self.state == 0

    def reset(self):
        self.state = self.n

    def __repr__(self):
        return f"NimBoard(palillos={self.state})"

In [ ]:
# Verificación rápida
board = NimBoard(n=15)
print("Estado inicial:", board)
print("Movimientos válidos:", board.valid_moves())
board.update(3)
print("Después de tomar 3:", board)
print("¿Partida terminada?", board.is_game_over())

## 2. El agente: clase `Agent`

El agente mantiene una **tabla de valores** `{estado -> valor}` donde el estado es simplemente el número de palillos restantes.

- En cada turno, evalúa los estados resultantes de cada acción válida y elige el de mayor valor (**explotación**).
- Con probabilidad `prob_exp` elige una acción al azar (**exploración**).
- Al recibir la recompensa al final de la partida, actualiza la tabla propagando el valor hacia atrás por todos los estados visitados.

In [ ]:
class Agent():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {}   # tabla: estado (int) -> valor estimado
        self.alpha = alpha          # tasa de aprendizaje
        self.states_visited = []   # historial de estados de la partida actual
        self.prob_exp = prob_exp   # probabilidad de exploración

    def reset(self):
        """Reinicia el historial al comienzo de cada partida."""
        self.states_visited = []

    def _get_value(self, state):
        """Devuelve el valor estimado de un estado (0.5 por defecto si es desconocido)."""
        return self.value_function.get(state, 0.5)

    def move(self, board, explore=True):
        """Decide qué acción tomar dado el estado actual del tablero."""
        valid = board.valid_moves()

        # Exploración: acción aleatoria
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            return np.random.choice(valid)

        # Explotación: elegir la acción que lleva al estado de mayor valor
        best_action = None
        best_value = -1
        for action in valid:
            next_state = board.state - action  # estado resultante
            value = self._get_value(next_state)
            if value > best_value:
                best_value = value
                best_action = action
        return best_action

    def update(self, board):
        """Registra el estado actual en el historial de la partida."""
        self.states_visited.append(board.state)

    def reward(self, reward):
        """
        Al final de la partida, propaga la recompensa hacia atrás por todos
        los estados visitados usando la actualización TD:
            V(S_t) <- V(S_t) + alpha * [recompensa - V(S_t)]
        """
        for state in reversed(self.states_visited):
            current = self._get_value(state)
            self.value_function[state] = current + self.alpha * (reward - current)
            reward = self.value_function[state]  # la recompensa se propaga hacia atrás

## 3. El juego: clase `Game`

La clase `Game` orquesta el entrenamiento: hace que dos agentes jueguen muchas partidas entre sí (`selfplay`) y gestiona las recompensas al final de cada una.

In [ ]:
from tqdm import tqdm


class Game():
    def __init__(self, player1, player2, n=15):
        self.players = [player1, player2]
        self.board = NimBoard(n=n)

    def selfplay(self, rounds=50000):
        """Hace jugar a los dos agentes `rounds` partidas y devuelve el conteo de victorias."""
        wins = [0, 0]
        draws = 0

        for _ in tqdm(range(rounds), desc="Entrenando"):
            self.board.reset()
            for player in self.players:
                player.reset()

            turn = 0
            winner_idx = None

            while not self.board.is_game_over():
                current_player = self.players[turn % 2]
                action = current_player.move(self.board)
                self.board.update(action)

                # Ambos jugadores registran el nuevo estado
                for player in self.players:
                    player.update(self.board)

                if self.board.is_game_over():
                    # Quien tomó el último palillo (turno actual) gana
                    winner_idx = turn % 2
                    break

                turn += 1

            # Asignar recompensas
            self._reward(winner_idx)

            if winner_idx is not None:
                wins[winner_idx] += 1

        print(f"\nResultados tras {rounds} partidas:")
        print(f"  Jugador 1 (agente): {wins[0]} victorias ({100*wins[0]/rounds:.1f}%)")
        print(f"  Jugador 2 (oponente): {wins[1]} victorias ({100*wins[1]/rounds:.1f}%)")
        return wins

    def _reward(self, winner_idx):
        """Da recompensa 1 al ganador y 0 al perdedor."""
        for i, player in enumerate(self.players):
            if i == winner_idx:
                player.reward(1.0)  # ganó
            else:
                player.reward(0.0)  # perdió

## 4. Entrenamiento

Entrenamos al agente con alta probabilidad de exploración al principio. Con 50 000 partidas, el espacio de estados es tan pequeño (solo 16 posibles valores de palillos) que converge rápidamente.

In [ ]:
N_PALILLOS = 15

# Agente principal (prob_exp alta al entrenar para explorar bien)
agente = Agent(alpha=0.5, prob_exp=0.4)

# Oponente de entrenamiento
oponente = Agent(alpha=0.5, prob_exp=0.4)

juego = Game(agente, oponente, n=N_PALILLOS)
resultados = juego.selfplay(rounds=50000)

## 5. Función de valor aprendida

Inspeccionamos qué aprendió el agente. El valor de cada estado indica la probabilidad estimada de ganar **desde ese número de palillos**.

La estrategia óptima matemática dice que son **posiciones perdedoras** para quien tiene que mover: 0, 4, 8, 12 (múltiplos de 4). Veremos si el agente lo aprendió.

In [ ]:
import pandas as pd

tabla = pd.DataFrame([
    {
        'palillos_restantes': state,
        'valor_estimado': round(value, 4),
        'es_multiplo_de_4': '✗ (perder)' if state % 4 == 0 else '✓ (ganar)'
    }
    for state, value in sorted(agente.value_function.items())
])

print("Función de valor aprendida por el agente:")
print(tabla.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

estados = sorted(agente.value_function.keys())
valores = [agente.value_function[s] for s in estados]
colores = ['#e74c3c' if s % 4 == 0 else '#2ecc71' for s in estados]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(estados, valores, color=colores, edgecolor='white', linewidth=0.8)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='valor inicial (0.5)')
ax.set_xlabel('Palillos restantes', fontsize=12)
ax.set_ylabel('Valor estimado (P(ganar))', fontsize=12)
ax.set_title('Función de valor aprendida — Nim con palillos', fontsize=14)
ax.set_xticks(estados)

from matplotlib.patches import Patch
leyenda = [Patch(color='#e74c3c', label='Múltiplo de 4 → posición perdedora'),
           Patch(color='#2ecc71', label='No múltiplo de 4 → posición ganadora')]
ax.legend(handles=leyenda, fontsize=10)

plt.tight_layout()
plt.show()

print("\nConclusión: valores bajos (≈0) en múltiplos de 4 → el agente aprendió")
print("que esas posiciones son desfavorables, tal como dicta la estrategia óptima.")

## 6. Guardar y cargar el agente entrenado

Guardamos la función de valor para poder reutilizarla sin volver a entrenar.

In [ ]:
import pickle

# Guardar
with open('agente_nim.pickle', 'wb') as f:
    pickle.dump(agente.value_function, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Agente guardado en 'agente_nim.pickle'")

In [ ]:
# Cargar (para usar en otra sesión)
with open('agente_nim.pickle', 'rb') as f:
    value_function_cargada = pickle.load(f)

agente_cargado = Agent(prob_exp=0.0)  # prob_exp=0: no explorar, solo explotar
agente_cargado.value_function = value_function_cargada

print(f"Agente cargado con {len(agente_cargado.value_function)} estados en su tabla.")

## 7. ¡A jugar! Tú vs. el agente entrenado

Ejecuta la siguiente celda para jugar una partida completa contra el agente.
- El tablero mostrará cuántos palillos quedan en cada turno.
- Cuando sea tu turno, ingresa cuántos palillos quieres tomar (1, 2 o 3).
- **Recuerda: quien toma el último palillo, ¡gana!**

> **Tip:** el agente juega sin exploración (`prob_exp=0`), usando solo lo que aprendió.

In [ ]:
def mostrar_palillos(n):
    """Muestra los palillos restantes de forma visual."""
    print(f"\n{'='*40}")
    print(f"  Palillos: {'| ' * n}")
    print(f"  Quedan {n} palillo{'s' if n != 1 else ''} en la mesa.")
    print(f"{'='*40}")


def jugar_contra_agente(agente_entrenado, n_palillos=15, humano_primero=True):
    """
    Partida interactiva: Humano vs. Agente entrenado.
    agente_entrenado : instancia de Agent ya entrenada (prob_exp=0 para no explorar)
    n_palillos       : palillos iniciales
    humano_primero   : True = el humano mueve primero
    """
    board = NimBoard(n=n_palillos)
    agente_entrenado.reset()

    turno_humano = humano_primero
    print("\n🎮 ¡Comienza la partida de Nim!")
    print("  Reglas: en cada turno toma 1, 2 o 3 palillos.")
    print("  Quien tome el ÚLTIMO palillo ¡GANA!")

    while not board.is_game_over():
        mostrar_palillos(board.state)
        validos = board.valid_moves()

        if turno_humano:
            # ── Turno del humano ──
            print(f"  🧑 Tu turno. Puedes tomar: {validos}")
            while True:
                try:
                    accion = int(input("  ¿Cuántos palillos tomas? "))
                    if accion in validos:
                        break
                    else:
                        print(f"  ⚠️  Número no válido. Elige entre {validos}.")
                except ValueError:
                    print("  ⚠️  Ingresa un número entero.")
            board.update(accion)
            agente_entrenado.update(board)  # el agente también registra el estado
            print(f"  → Tomaste {accion} palillo{'s' if accion != 1 else ''}.")
        else:
            # ── Turno del agente ──
            print("  🤖 Turno del agente...")
            accion = agente_entrenado.move(board, explore=False)
            board.update(accion)
            agente_entrenado.update(board)
            print(f"  → El agente tomó {accion} palillo{'s' if accion != 1 else ''}.")

        turno_humano = not turno_humano  # cambiar turno

    # Fin del juego
    mostrar_palillos(0)
    # El último en mover fue quien NO tiene el turno actual
    gano_humano = turno_humano  # si ahora es turno del humano, el agente fue el último en mover... espera
    # Lógica: turno_humano se invierte DESPUÉS de cada movimiento.
    # Si tras el último movimiento es turno del humano → el agente fue el último en mover → agente ganó
    # Si tras el último movimiento es turno del agente → el humano fue el último en mover → humano ganó
    if turno_humano:
        print("\n🤖 ¡El AGENTE ganó! Mejor suerte la próxima vez. 😄")
    else:
        print("\n🎉 ¡GANASTE! ¡Enhorabuena, venciste al agente! 🏆")


# ── Lanzar la partida ──────────────────────────────────────────────────────────
agente_jugador = Agent(prob_exp=0.0)
agente_jugador.value_function = agente.value_function.copy()

jugar_contra_agente(agente_jugador, n_palillos=N_PALILLOS, humano_primero=True)

## 8. Evaluación: ¿qué tan bueno es el agente?

Comparamos el agente entrenado contra un jugador **aleatorio** para cuantificar cuánto aprendió.

In [ ]:
class RandomAgent():
    """Jugador que siempre elige un movimiento al azar (línea base)."""
    def reset(self):
        pass

    def move(self, board, explore=True):
        return np.random.choice(board.valid_moves())

    def update(self, board):
        pass

    def reward(self, reward):
        pass


def evaluar(agente_entrenado, n_partidas=1000, n_palillos=15):
    """Juega n_partidas del agente entrenado (prob_exp=0) contra un agente aleatorio."""
    agente_entrenado.prob_exp = 0.0  # solo explotación durante evaluación
    aleatorio = RandomAgent()

    wins_agente = 0
    for _ in range(n_partidas):
        board = NimBoard(n=n_palillos)
        agente_entrenado.reset()
        turno = 0
        ganador = None

        while not board.is_game_over():
            if turno % 2 == 0:
                action = agente_entrenado.move(board, explore=False)
            else:
                action = aleatorio.move(board)
            board.update(action)
            if board.is_game_over():
                ganador = turno % 2  # 0 = agente, 1 = aleatorio
            turno += 1

        if ganador == 0:
            wins_agente += 1

    tasa = 100 * wins_agente / n_partidas
    print(f"Evaluación ({n_partidas} partidas, agente mueve primero):")
    print(f"  Victorias del agente entrenado: {wins_agente} ({tasa:.1f}%)")
    print(f"  Victorias del agente aleatorio: {n_partidas - wins_agente} ({100-tasa:.1f}%)")
    print()
    if N_PALILLOS % 4 == 0:
        print(f"  ℹ️  Con N={N_PALILLOS} (múltiplo de 4), el primer jugador tiene DESVENTAJA.")
        print("     El agente óptimo debería ganar < 50% cuando mueve primero.")
    else:
        print(f"  ℹ️  Con N={N_PALILLOS} (no múltiplo de 4), el primer jugador tiene VENTAJA.")
        print("     El agente óptimo debería ganar > 50% cuando mueve primero.")


evaluar(agente, n_partidas=2000, n_palillos=N_PALILLOS)

## 9. Curva de aprendizaje

Visualizamos cómo mejora el agente a lo largo del entrenamiento, calculando su tasa de victorias en intervalos regulares.

In [ ]:
def curva_de_aprendizaje(n_palillos=15, rondas_total=50000, intervalo=2000):
    """Entrena un agente y registra su tasa de victorias contra un oponente aleatorio."""
    ag = Agent(alpha=0.5, prob_exp=0.4)
    rival = Agent(alpha=0.5, prob_exp=0.4)
    board = NimBoard(n=n_palillos)

    checkpoints = []
    tasas = []

    for ronda in tqdm(range(1, rondas_total + 1), desc="Calculando curva"):
        # Partida de entrenamiento
        board.reset()
        ag.reset()
        rival.reset()
        turno = 0
        ganador = None
        while not board.is_game_over():
            jugador_actual = ag if turno % 2 == 0 else rival
            action = jugador_actual.move(board)
            board.update(action)
            ag.update(board)
            rival.update(board)
            if board.is_game_over():
                ganador = turno % 2
            turno += 1
        ag.reward(1.0 if ganador == 0 else 0.0)
        rival.reward(1.0 if ganador == 1 else 0.0)

        # Evaluación cada `intervalo` rondas
        if ronda % intervalo == 0:
            victorias = 0
            evaluaciones = 200
            ag.prob_exp = 0.0
            for _ in range(evaluaciones):
                board.reset()
                ag.reset()
                turno = 0
                ganador = None
                while not board.is_game_over():
                    if turno % 2 == 0:
                        action = ag.move(board, explore=False)
                    else:
                        action = np.random.choice(board.valid_moves())
                    board.update(action)
                    if board.is_game_over():
                        ganador = turno % 2
                    turno += 1
                if ganador == 0:
                    victorias += 1
            ag.prob_exp = 0.4  # restaurar exploración
            checkpoints.append(ronda)
            tasas.append(100 * victorias / evaluaciones)

    # Graficar
    plt.figure(figsize=(10, 4))
    plt.plot(checkpoints, tasas, color='#3498db', linewidth=2, marker='o', markersize=4)
    plt.axhline(50, color='gray', linestyle='--', linewidth=1, label='50% (azar)')
    plt.xlabel('Partidas de entrenamiento', fontsize=12)
    plt.ylabel('Victorias vs. jugador aleatorio (%)', fontsize=12)
    plt.title('Curva de aprendizaje — Nim con palillos', fontsize=14)
    plt.legend()
    plt.ylim(0, 105)
    plt.tight_layout()
    plt.show()


curva_de_aprendizaje(n_palillos=N_PALILLOS)

## Resumen

En este cuadernillo hemos aplicado los mismos principios del axr que en el ejemplo del tres en raya:

| Concepto | Tres en raya | Nim con palillos |
|---|---|---|
| **Estado** | Configuración del tablero 3×3 | Número de palillos restantes |
| **Acciones** | Colocar X/O en celda vacía | Tomar 1, 2 o 3 palillos |
| **Función de valor** | Tabla `{estado_string -> valor}` | Tabla `{entero -> valor}` |
| **Actualización** | TD hacia atrás al final de la partida | Idéntica |
| **Exploración** | `prob_exp` aleatoria | Idéntica |
| **Verificación** | Heurística / intuición | Estrategia óptima conocida (mod 4) |

El hecho de que el espacio de estados de Nim sea muy pequeño (solo 16 posibles valores) permite que el agente converja rápidamente y que podamos **verificar matemáticamente** que aprendió la estrategia óptima: asignar valores bajos a las posiciones múltiplos de 4.